# НИС «Основы анализа данных в Python»

*Алла Тамбовцева*

## Практикум 17. Логистическая регрессия: часть 2

### Описание данных и постановка задачи

В файле `flowers_two.csv` хранятся характеристики 186 фотографий, на которых изображены цветы:

* `R`: средняя интенсивность красного цвета (усредненное значение по всем пикселям);
* `G`: средняя интенсивность зеленого цвета;
* `B`: средняя интенсивность синего цвета;
* `Name`: название цветка (`foxglove` – наперстянка, `monkshood` – аконит).

Источник изображений – [Kaggle](https://www.kaggle.com/datasets/yousefmohamed20/oxford-102-flower-dataset).

**Задача** 

Научиться классифицировать изображения по их цветовым характеристикам (здесь упрощенный вариант, поскольку файлы с изображениями уже обработаны и мы имеем дело с обычным датафреймом привычной размерности).

### Задания

Импортируем все необходимые библиотеки и функции:

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import log_loss, roc_curve, roc_auc_score

from matplotlib import pyplot as plt

Загрузим данные:

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/allatambov/PyDat25/refs/heads/main/flowers_two.csv")
df.head()

### Задача 0: оценка модели

a. Закодируйте признак `Name` при помощи `LabelEncoder`, результат сохраните в переменную `Class`. Рассчитайте среднее значение полученного признака.

b. Постройте модель логистической регрессии:

$$
\hat{P}(y = 1) = \sigma(\hat{\omega}_0 + \hat{\omega}_1 \times R + \hat{\omega}_2 \times B), 
$$

где $y$ – признак, получившийся при кодировании в предыдущей задаче.

Для этого разделите выборку на тренировочную и тестовую выборку в соотношении 80 к 20, воспроизводимость (`random_state`) равна 24 и обучите модель на тренировочной выборке.

In [ ]:
le = LabelEncoder()
df["Class"] = le.fit_transform(df["Name"])

print("Доля 1:", df["Class"].mean())
print("Доля 0:", 1 - df["Class"].mean())

In [ ]:
X = df[["R", "B"]]
y = df["Class"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size = 0.2, 
                                                    random_state = 42)

In [ ]:
### YOUR CODE HERE ###

### Задача 1: коэффициенты модели

Выведите коэффициенты модели – оценки коэффициентов при `R` и `B`. Преобразуйте их и проинтерпретируйте в терминах изменения отношения шансов.

In [ ]:
### YOUR CODE HERE ###

### Задача 2: метрики качества, зависящие от порогового значения вероятности

Постройте матрицу ошибок и оцените качество модели на тестовой выборке, вычислив:

* точность модели (*accuracy*)
* точность модели (*precision* или *predictive positive value*)
* чувствительность модели (полнота *recall* или *true positive rate*, *TPR*)
* специфичность модели (*true negative rate*, *TNR*)
* F1-меру для модели

Полный набор метрик с альтернативными названиями – см [таблицу](https://en.wikipedia.org/wiki/Confusion_matrix).

In [ ]:
# создайте массив y_pred с предсказанными значениями y^

### YOUR CODE HERE ###

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(f"Precision (PPV): {precision_score(y_test, y_pred):.2f}")

### YOUR CODE HERE ###

### Задача 3: метрики качества, не зависящие от порогового значения вероятности

Оцените качество модели на тестовой выборке:

* вычислите значение логистической функции потерь
* вычислите значение метрики AUC – сохраните ее в переменную `auc`

In [ ]:
# создайте массив с предсказанными значениями вероятностей,
# заберите оттуда вероятности 1 и сохраните как p_hat

### YOUR CODE HERE ###

Постройте ROC-кривую для модели и подпишите на нем значение AUC.

In [ ]:
fpr, tpr, cutoffs = roc_curve(y_test, p_hat)

print(fpr)
print(tpr)
print(cutoffs)

In [ ]:
plt.plot(fpr, tpr, 
         color = "navy", 
         label = f"Model (AUC = {auc:.2f})");

plt.plot([0, 1], [0, 1], 
         color = "red", 
         linestyle = "dashed", 
         label = "Random classifier");

plt.grid();
plt.legend(loc = "lower right");
plt.xlabel("FPR: 1 – Specificity");
plt.ylabel("TPR: Sensitivity");
plt.show()

### Задача 4: изменение пороговой вероятности

Постройте график, где по горизонтальной оси указаны пороговые значения вероятности `cutoffs`, а по вертикальной оси – значения чувствительности (`tpr`) и специфичности (`1-fpr`).

In [ ]:
# чтобы не запутаться, переименуем tpr и вычислим 1 - fpr явно

sensitivity_vals = tpr
specificity_vals = 1 - fpr

### YOUR CODE HERE ###

plt.legend(loc = "center right");
plt.xlabel("Cutoff points")
plt.grid();
plt.show()

Определите, какой порог отсечения стоит брать, чтобы чувствительность и специфичность метода одновременно достигали максимально возможных значений.

In [ ]:
# когда достигается максимум среднего геометрического метрик?

geom_mean = (sensitivity_vals * specificity_vals) ** 0.5
print(geom_mean)

### YOUR CODE HERE ###

Переоцените модель на обучающей выборке с измененным пороговым значением и проверьте ее качество на тестовой выборке, вычислив метрики, зависящие от порогового значения вероятности.

In [ ]:
# измените cutoff на новое значение пороговой вероятности

y_pred_new = (p_hat > cutoff).astype(int)

### YOUR CODE HERE ###

print(f"Accuracy: {accuracy_score(y_test, y_pred_new):.2f}")
print(f"Precision (PPV): {precision_score(y_test, y_pred_new):.2f}")
print(f"Sensitivity (TPR): {recall_score(y_test, y_pred_new):.2f}")
print(f"Specificity (TNR): {recall_score(y_test, y_pred_new, pos_label = 0):.2f}")
print(f"F1: {f1_score(y_test, y_pred_new):.2f}")